# 00. Multi-Cohort Diabetic Retinopathy Harmonization Walkthrough

This notebook demonstrates the unified data ingestion, circular Field-of-View (FOV) segmentation, green-channel enhancement, and patient namespacing across the six clinical cohorts:
- **APTOS 2019** (India)
- **DDR** (China)
- **DeepDRiD** (China)
- **IDRiD** (India)
- **Messidor-2** (France)
- **EyePACS** (USA)


In [ ]:
import numpy as np
import pandas as pd
from drdg.data.registry import COHORT_REGISTRY
from drdg.data.metadata import MasterFundusRecord
from drdg.data.fov import compute_fov_mask, apply_fov_mask
from drdg.data.preprocessing import crop_fundus_roi, extract_green_channel, apply_ben_graham_enhancement
from drdg.data.patient_ids import parse_patient_and_side

print(f"Registered cohorts in DRDG: {list(COHORT_REGISTRY.keys())}")


## 1. Inspect Cohort Metadata Registry

In [ ]:
registry_summary = []
for key, spec in COHORT_REGISTRY.items():
    registry_summary.append({
        "Cohort": spec.display_name,
        "Country": spec.country,
        "Camera": spec.camera_model,
        "FOV (deg)": spec.fov_degrees,
        "Grading Protocol": spec.grading_protocol
    })
pd.DataFrame(registry_summary)


## 2. Preprocessing & Morphological FOV Extraction

In [ ]:
# Synthesize a simulated fundus photograph with peripheral black borders
synthetic_fundus = np.zeros((400, 400, 3), dtype=np.uint8)
y, x = np.ogrid[:400, :400]
disk_mask = ((x - 200)**2 + (y - 200)**2) <= 160**2
synthetic_fundus[disk_mask] = [80, 160, 210]  # Retinal tissue

# 1. Non-black ROI crop
cropped = crop_fundus_roi(synthetic_fundus)

# 2. 4-step Morphological FOV segmentation
fov_mask = compute_fov_mask(cropped)

# 3. Green channel extraction (highest lesion contrast)
green = extract_green_channel(cropped)

print(f"Original shape: {synthetic_fundus.shape}, Cropped ROI: {cropped.shape}, FOV mask: {fov_mask.shape}")


## 3. Standardized Patient Namespacing

In [ ]:
test_cases = [
    ("001", "aptos"),
    ("001-0001-000.jpg", "ddr"),
    ("001_l1.jpg", "deepdrid"),
    ("IDRiD_042.jpg", "idrid"),
    ("20051019_38557_0100_PP.tif", "messidor2"),
    ("10_left.jpeg", "eyepacs"),
]

for img_id, cohort in test_cases:
    pid, side = parse_patient_and_side(img_id, cohort)
    print(f"Cohort: {cohort:10s} | Image: {img_id:30s} -> Patient ID: {pid:20s} | Side: {side}")
